# Agentic Cluster Workflow: Letting an AI Agent Run Explorer for You

AI coding agents (Claude Code, Codex, etc.) can SSH into Explorer, submit jobs, watch them, and
report back -- so you stop babysitting a terminal. This notebook walks through the same setup an
agent needs, step by step, so you understand exactly what it's doing on your behalf. Everything
here is also encoded in this repo's [`SKILL.md`](../SKILL.md), which an agent reads automatically.

**What you'll set up:**
1. Passwordless SSH (the thing that makes automation possible at all)
2. A project `.env` for secrets, correctly gitignored
3. Your own project `SKILL.md`, generated from the repo template
4. A short agent-style loop: submit &rarr; poll &rarr; tail log &rarr; report
5. (Optional) Weights & Biases run status pulled the same way

Run the shell cells here (`!command`) from your **laptop** or from an always-on box you SSH from --
not from inside a compute node.

## 1. Passwordless SSH

An agent cannot type a password or approve a Duo/2FA push. Key-based auth is what makes
`ssh explorer '...'` usable in a script or by an agent without a human present.

Run these once, from your local machine (edit `your-username` first):

In [ ]:
# 1a. generate a key (no passphrase -- needed for unattended use)
!ssh-keygen -t ed25519 -N "" -f ~/.ssh/id_ed25519 -q || echo "key already exists, skipping"


In [ ]:
# 1b. add a friendly host alias so you (and the agent) can just say `explorer`
alias_block = '''
Host explorer
    HostName login.explorer.northeastern.edu
    User your-username
    ServerAliveInterval 60
'''
import pathlib
cfg = pathlib.Path.home() / ".ssh" / "config"
existing = cfg.read_text() if cfg.exists() else ""
if "Host explorer" not in existing:
    with open(cfg, "a") as f:
        f.write(alias_block)
    cfg.chmod(0o600)
    print("added `explorer` alias to ~/.ssh/config -- edit the User line with your username")
else:
    print("`explorer` alias already present in ~/.ssh/config")


In [ ]:
# 1c. install your public key on Explorer (asks for your NEU password ONE more time)
!ssh-copy-id -i ~/.ssh/id_ed25519.pub your-username@login.explorer.northeastern.edu


In [ ]:
# 1d. verify -- should print instantly, no prompt
!ssh explorer 'echo passwordless works && hostname'


## 2. Secrets: `.env` + `.gitignore`

Never put tokens/passwords in a notebook cell, a prompt to your agent, or a commit. Put them in a
project-root `.env` and gitignore it **first**.

In [ ]:
import pathlib

project_root = pathlib.Path(".").resolve()   # run this from your project root
gitignore = project_root / ".gitignore"
env_line = ".env\n"

text = gitignore.read_text() if gitignore.exists() else ""
if ".env" not in text.split():
    with open(gitignore, "a") as f:
        f.write(env_line)
    print("added .env to .gitignore")
else:
    print(".env already gitignored")

env_path = project_root / ".env"
if not env_path.exists():
    env_path.write_text(
        "WANDB_API_KEY=\n"
        "GITHUB_TOKEN=\n"
        "# EXPLORER_SSH_KEY=   # only needed to re-install the pubkey if it's ever lost\n"
    )
    env_path.chmod(0o600)
    print("created .env template -- fill in your real values, never commit it")
else:
    print(".env already exists -- leaving it alone")


## 3. Generate Your Project's `SKILL.md`

Copy this repo's [`SKILL.md`](../SKILL.md) into your own project and fill in the placeholders.
An agent (or you, later) reads this file first and immediately knows how your project's cluster
workflow works -- no re-explaining every session.

In [ ]:
import shutil, pathlib

template = pathlib.Path("../SKILL.md")          # this repo's generic template
dest_dir = pathlib.Path(".claude/skills")        # project-local skill (see template's own notes)
dest_dir.mkdir(parents=True, exist_ok=True)
dest = dest_dir / "SKILL.md"

if template.exists() and not dest.exists():
    shutil.copy(template, dest)
    print(f"copied template to {dest} -- now edit the placeholders (your-username, your-project, paths)")
elif dest.exists():
    print(f"{dest} already exists -- leaving it alone")
else:
    print("template SKILL.md not found relative to this notebook -- adjust the path")


## 4. Agent-Style Job Loop: Submit &rarr; Poll &rarr; Tail &rarr; Report

This is the same loop an agent runs when you ask it to "submit the training job and tell me when
it's done." Here it is as plain Python + `subprocess`, so you can see exactly what's happening.

Adjust `REMOTE_DIR` and the `sbatch` script name for your own project before running for real.

In [ ]:
import subprocess

def ssh(cmd: str) -> str:
    """Run a command on Explorer over the passwordless alias and return stdout."""
    result = subprocess.run(["ssh", "explorer", cmd], capture_output=True, text=True, timeout=60)
    if result.returncode != 0:
        raise RuntimeError(f"remote command failed: {result.stderr}")
    return result.stdout

REMOTE_DIR = "~/your-project"  # edit me


In [ ]:
# submit (uncomment when you have a real sbatch script)
# out = ssh(f"cd {REMOTE_DIR} && sbatch --parsable scripts/train.sbatch")
# job_id = out.strip()
# print("submitted job", job_id)


In [ ]:
# poll status
def job_status(job_id: str) -> str:
    return ssh(f'squeue -j {job_id} --format="%.10i %.2t %.12M %R" --noheader') or "(not in queue -- finished or not yet scheduled)"

# print(job_status(job_id))


In [ ]:
# tail the log once running
def tail_log(path: str, n: int = 40) -> str:
    return ssh(f"tail -{n} {path}")

# print(tail_log(f"{REMOTE_DIR}/logs/train_{job_id}.out"))


In [ ]:
# after it ends: what happened
def job_summary(job_id: str) -> str:
    return ssh(f"sacct -j {job_id} --format=JobID,JobName,State,ExitCode,Elapsed")

# print(job_summary(job_id))


In an actual agentic session you'd just ask your agent: *"submit `scripts/train.sbatch` and
let me know when it finishes or fails."* It runs the same four functions above in a loop, decides
when to stop polling, and summarizes the SLURM state (and the log tail) back to you in plain
English.

**Cluster is often busy -- prefer several short jobs chained with `--dependency` over one long
one:**
```bash
j1=$(sbatch --parsable scripts/stage1_prep.sbatch)
j2=$(sbatch --parsable --dependency=afterok:$j1 scripts/stage2_train.sbatch)
```
Checkpoint periodically inside `stage2_train.sbatch` so a killed job resumes instead of
restarting.

## 5. (Optional) Pull Weights & Biases Run Status the Same Way

With `WANDB_API_KEY` set in `.env` and your training script calling `wandb.init(...)` /
`wandb.log(...)` as usual, you (or your agent) can pull the latest metrics without opening the
dashboard: 

In [ ]:
import os

# load .env into the environment (simple manual loader -- avoids adding a dependency)
def load_dotenv(path=".env"):
    import pathlib
    p = pathlib.Path(path)
    if not p.exists():
        return
    for line in p.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        if v:
            os.environ.setdefault(k, v)

load_dotenv()

# import wandb
# api = wandb.Api()
# run = api.run("your-entity/your-project/your-run-id")
# print(run.summary)   # latest logged metrics, e.g. {'loss': 0.42, 'accuracy': 0.91, ...}
print("WANDB_API_KEY set:", bool(os.environ.get("WANDB_API_KEY")))


## Summary

You now have everything an agent needs to operate Explorer for this project hands-off:

- Passwordless SSH (`ssh explorer '...'` works with no prompts)
- A gitignored `.env` holding your tokens
- A project `SKILL.md` describing your specific workflow
- The submit/poll/tail/report loop an agent runs on your behalf
- (Optional) W&B run status pulled programmatically

Next: see `sbatch_job_submission_guide.ipynb` and `srun_interactive_gpu_guide.ipynb` for the SLURM
side of things, and this repo's `SKILL.md` for the checked-in version of everything above.